In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
!pip install python-doctr


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 21.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.4/288.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.2 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=cf4c4c3ecd1073963f7816f561b9ece7a29d677432f1f1f51b193aed49318bf0
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1

In [ ]:
# LOAD DOCTR + BASE TrOCR HANDWRITTEN
def load_models():
    import torch
    from doctr.models import ocr_predictor
    from transformers import TrOCRProcessor, VisionEncoderDecoderModel

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Using device:", device)

    print("Loading DocTR line detector...")
    detection_model = ocr_predictor(pretrained=True).to(device).eval()

    # BASE TrOCR Large Handwritten
    print("Loading BASE TrOCR Large Handwritten...")
    base_model_name = "microsoft/trocr-large-handwritten"
    base_processor = TrOCRProcessor.from_pretrained(base_model_name)
    base_model = VisionEncoderDecoderModel.from_pretrained(base_model_name).to(device).eval()

    print("✔ Models loaded successfully")
    # Only return base model components
    return detection_model, base_processor, base_model, device

In [ ]:
detection_model, base_processor, base_model, device = load_models()

Using device: cpu
Loading DocTR line detector...


  0%|          | 0/65814772 [00:00<?, ?it/s]

  0%|          | 0/63303144 [00:00<?, ?it/s]

Loading BASE TrOCR Large Handwritten...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

pytorch_model.bin:   0%|          | 0.00/2.23G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.23G [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-large-handwritten and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✔ Models loaded successfully


In [ ]:


import cv2
import torch
import matplotlib.pyplot as plt

def run_inference(image_path, detection_model, processor, trocr_model, device, model_name="TrOCR"):
    import cv2
    import torch
    import matplotlib.pyplot as plt
    from statistics import median

    # Load image
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img_gray_rgb = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)
    height, width = img_rgb.shape[:2]

    # DocTR line detection
    result = detection_model([img_gray_rgb])
    results = []
    line_count = 0
    boxed_img = img_rgb.copy()

    for page in result.pages:
        for block in page.blocks:
            for line in block.lines:
                (x0, y0), (x1, y1) = line.geometry
                x_min = int(x0 * width)
                y_min = int(y0 * height)
                x_max = int(x1 * width)
                y_max = int(y1 * height)

                # Padding
                pad = 5
                x_min = max(0, x_min - pad)
                y_min = max(0, y_min - pad)
                x_max = min(width, x_max + pad)
                y_max = min(height, y_max + pad)

                crop = img_gray_rgb[y_min:y_max, x_min:x_max]
                if crop.shape[0] < 5 or crop.shape[1] < 5:
                    continue

                line_count += 1

                # TrOCR prediction
                pixel_values = processor(images=crop, return_tensors="pt").pixel_values.to(device)
                with torch.no_grad():
                    generated_ids = trocr_model.generate(pixel_values)
                text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

                y_center = (y_min + y_max) / 2.0

                results.append({
                    "line_number": line_count,
                    "bbox": (x_min, y_min, x_max, y_max),
                    "y_center": y_center,
                    "text": text
                })

                # Draw bounding box
                cv2.rectangle(boxed_img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

                # === PER-LINE VISUAL CHECK RESTORED ===
                plt.figure(figsize=(14, 2))
                plt.imshow(crop)
                plt.title(f"{model_name} - Box {line_count}: {text}")
                plt.axis('off')
                plt.show()

                print(f"[{model_name} - Box {line_count}] {text}")
                print(f"BBox: {x_min,y_min,x_max,y_max}\n")

    # Sort lines
    results_sorted = sorted(results, key=lambda r: (r["y_center"], r["bbox"][0]))
    heights = [r["bbox"][3]-r["bbox"][1] for r in results_sorted]
    if heights:
        median_h = median(heights)
        clustered = []
        current_cluster = [results_sorted[0]]
        for r in results_sorted[1:]:
            if abs(r["y_center"] - current_cluster[-1]["y_center"]) <= (median_h * 0.5):
                current_cluster.append(r)
            else:
                clustered.extend(sorted(current_cluster, key=lambda x: x["bbox"][0]))
                current_cluster = [r]
        clustered.extend(sorted(current_cluster, key=lambda x: x["bbox"][0]))
        results_sorted = clustered

    # Build final paragraph
    paragraph = " ".join([r["text"].strip() for r in results_sorted if r["text"].strip()]).strip()

    # Show final boxed image
    plt.figure(figsize=(12, 12))
    plt.imshow(boxed_img)
    plt.title(f"Detected Text with Bounding Boxes - {model_name}")
    plt.axis('off')
    plt.show()

    print(f"FINAL RECOGNIZED PARAGRAPH ({model_name}):\n")
    print(paragraph)

    return results_sorted, paragraph






In [ ]:
image_path =  "/content/drive/MyDrive/1dYPfu_k2Ibjfh8LuUVJLnd7vICBeYDQ1"




In [ ]:
# Run inference with FINETUNED TrOCR
# This section is commented out as the user requested not to load the finetuned model.
# finetuned_results, finetuned_paragraph = run_inference(
#     image_path, detection_model, finetuned_processor, finetuned_model, device, model_name="Finetuned TrOCR"
# )

In [ ]:
# # Run inference with BASE TrOCR Handwritten
# base_results, base_paragraph = run_inference(
#     image_path, detection_model, base_processor, base_model, device, model_name="Base TrOCR Handwritten"
# )

In [ ]:
image_path = "/content/10.jpg"
# Alternatively, if you have a file in Google Drive, ensure the path is correct and the drive is mounted:
# image_path = "/content/drive/MyDrive/Your_Image_Name.jpg"


In [ ]:
# 1. List contents of the current directory
import os
print("Contents of current directory:")
print(os.listdir('.'))


Contents of current directory:
['.config', 'drive', 'sample_data']


In [ ]:
# 2. List contents of a specific folder (e.g., '/content/drive/MyDrive')
# Make sure Google Drive is mounted if you're trying to access it.
import os
folder_path = '/content/drive/MyDrive'
if os.path.exists(folder_path):
    print(f"\nContents of '{folder_path}':")
    print(os.listdir(folder_path))
else:
    print(f"\nFolder '{folder_path}' does not exist or is not mounted.")



Contents of '/content/drive/MyDrive':
['Colab Notebooks', 'resume.pdf', 'Classroom', '2. IICT lab2 (1).docx.gdoc', '2. IICT lab2.docx.gdoc', 'abubakar.docx', ' MATHS Calculus notes ', 'Thomas_Calculus (1).gdoc', 'Assignment 1  (1).gdoc', 'Thomas_Calculus.gdoc', 'Section 1.3 and 1.4.gslides', 'Assignment 1 .gdoc', 'Mathematica.gdoc', 'English Assignment Task 1 (2).docx', 'English Assignment Task 1 (1).docx', 'English Assignment Task 1.docx', 'Lab04 (3).gdoc', 'Lab04 (2).gdoc', 'Lumio', 'Question bank.gdoc', 'pakistan studies assighnment 1.docx', 'drinks (1).gsheet', 'LabManual_Excel (3).gdoc', 'AttendanceSheet (3).gsheet', 'LabManual_Excel (2).gdoc', 'LabManual_Excel (1).gdoc', 'AttendanceSheet (2).gsheet', 'drinks.gsheet', 'AttendanceSheet (1).gsheet', 'LabManual_Excel.gdoc', 'AttendanceSheet.gsheet', 'Starting Out with Cpp from Control Structures to Objects 8th Edition By Tony Gaddis.gdoc', 'list of practice questions.gdoc', 'Trignometry.gdoc', 'limit of a function (1).gdoc', 'lab6.g

In [ ]:
# 3. Check if a specific path exists
import os
check_path = '/content/drive/MyDrive/dlsathvik04 Dyslexia_Detection main data-dyslexic/10.jpg'
if os.path.exists(check_path):
    print(f"\nPath '{check_path}' exists.")
else:
    print(f"\nPath '{check_path}' does not exist.")

check_path_folder = '/content/drive/MyDrive/dlsathvik04 Dyslexia_Detection main data-dyslexic'
if os.path.exists(check_path_folder):
    print(f"Path '{check_path_folder}' exists.")
else:
    print(f"Path '{check_path_folder}' does not exist.")



Path '/content/drive/MyDrive/dlsathvik04 Dyslexia_Detection main data-dyslexic/10.jpg' exists.
Path '/content/drive/MyDrive/dlsathvik04 Dyslexia_Detection main data-dyslexic' exists.


In [ ]:
# 4. Print current working directory
import os
print(f"\nCurrent working directory: {os.getcwd()}")



Current working directory: /content


In [ ]:
# Run inference with BASE TrOCR Handwritten
#base_results, base_paragraph = run_inference(
 #   image_path, detection_model, base_processor, base_model, device, model_name="Base TrOCR Handwritten"
#)


In [ ]:
image_path = "/content/drive/MyDrive/dlsathvik04 Dyslexia_Detection main data-dyslexic/10.jpg"

In [ ]:
# Run inference with BASE TrOCR Handwritten
base_results, base_paragraph = run_inference(
    image_path, detection_model, base_processor, base_model, device, model_name="Base TrOCR Handwritten"
)

error: OpenCV(4.12.0) /io/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'


In [ ]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# # 1. Point to your folder
# model_path = "./DyslexAI_Best_Model"  # Make sure this matches your folder name

# print("Loading your custom model...")
# tokenizer = AutoTokenizer.from_pretrained(model_path)
# model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# # 2. Test it!
# text = "correct dyslexia: I hav been working on this proejct for to long."

# inputs = tokenizer(text, return_tensors="pt")
# outputs = model.generate(**inputs, max_length=64)
# correction = tokenizer.decode(outputs[0], skip_special_tokens=True)

# print(f"Original: {text}")
# print(f"Correction: {correction}")

In [ ]:
# First, let's make sure Google Drive is mounted. It seems it is already mounted from previous steps.

import os
import zipfile
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Define the path to your zipped model in Google Drive
zipped_model_path = "/content/drive/MyDrive/DyslexAI_Best_Model (1).zip"

# Define the directory where you want to extract the model
extraction_path = "./DyslexAI_Best_Best_Model_unzipped"

# Create the extraction directory if it doesn't exist
os.makedirs(extraction_path, exist_ok=True)

print(f"Unzipping model from {zipped_model_path} to {extraction_path}...")

# Unzip the model
with zipfile.ZipFile(zipped_model_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print("✔ Model unzipped successfully.")

# Load the tokenizer and model from the unzipped directory
print("Loading your custom model from unzipped folder...")
tokenizer = AutoTokenizer.from_pretrained(extraction_path)
model = AutoModelForSeq2SeqLM.from_pretrained(extraction_path)

print("✔ Custom model loaded successfully.")


Unzipping model from /content/drive/MyDrive/DyslexAI_Best_Model (1).zip to ./DyslexAI_Best_Best_Model_unzipped...
✔ Model unzipped successfully.
Loading your custom model from unzipped folder...
✔ Custom model loaded successfully.


In [ ]:
# 2. Test the loaded model
text = "correct dyslexia: I hav been working on this proejct for to long."

inputs = tokenizer(text, return_tensors="pt")
outputs = model.generate(**inputs, max_length=64)
correction = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Original: {text}")
print(f"Correction: {correction}")


Original: correct dyslexia: I hav been working on this proejct for to long.
Correction: I have been working on this project for too long.


In [ ]:
pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.0 MB/s eta 0:00:00


In [ ]:
import os
import re
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from groq import Groq  # <--- CHANGED: Import Groq

# ==========================================
# 1. SETUP API (GROQ)
# ==========================================
# Your Groq Key
GROQ_API_KEY = "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"

# Initialize Groq Client
client = Groq(api_key=GROQ_API_KEY)

def get_groq_correction(text):
    """
    Uses Groq (Llama 3.3) to fix context errors that the local model missed.
    """
    print("📡 Contacting Groq (Llama 3.3) Layer...")

    system_prompt = """
    You are an expert English Editor specializing in fixing OCR errors and dyslexic text.

    Your Task:
    1. Fix spelling and grammatical errors based on context.
    2. Specifically look for historical or proper noun errors (e.g., "13 Stakes" -> "13 States", "Nklymuh" -> "Nkrumah").
    3. Do NOT add conversational filler. Output ONLY the corrected text.
    """

    try:
        completion = client.chat.completions.create(
            # UPDATED MODEL ID HERE:
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": text
                }
            ],
            temperature=0.1, # Lowered slightly for more precision
            max_tokens=1024,
            top_p=1,
            stop=None,
            stream=False
        )

        return completion.choices[0].message.content.strip()

    except Exception as e:
        print(f"❌ Groq API Error: {e}")
        return text # Return previous step text if API fails

# ==========================================
# 2. LOAD LOCAL JANITOR (Your T5)
# ==========================================
# Check if model exists locally or needs to be loaded
if 'model' not in globals():
    # Update this list with the actual path to your unzipped folder
    possible_paths = ["DyslexAI_Model_Unzipped", "DyslexAI_Best_Model (1)", "/content/DyslexAI_Model_Unzipped"]

    # Simple check to find the first valid path
    path = next((p for p in possible_paths if os.path.exists(p)), None)

    if path:
        print(f"⏳ Loading Local Model from {path}...")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(path)
        model = AutoModelForSeq2SeqLM.from_pretrained(path).to(device)
    else:
        print("⚠️ Local model path not found. Please ensure your T5 model is unzipped.")
        # Create dummy placeholders if model isn't found so code doesn't crash during testing
        model = None
        tokenizer = None

# ==========================================
# 3. THE PIPELINE EXECUTION
# ==========================================
def layer_1_sanitize(text):
    """Basic Regex Cleaning"""
    text = text.replace(" . ", " ")
    text = re.sub(r'[^a-zA-Z0-9\s.,!?\'"-]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def layer_2_dyslexia_fix(text):
    """Local T5 Model Fix"""
    if model is None: return text # Skip if model wasn't loaded

    input_text = "correct dyslexia: " + text
    device = model.device
    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=512, num_beams=2, early_stopping=True)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ==========================================
# 4. RUN FINAL TEST
# ==========================================
messy_input = 'if the threatened . counter . revolution " not # to bring the President . backs . thes . 13 Stakes . of the Commonwealth was an occasion worthy . of his . presence . after . all it . was Mr. Nklymuh'

print("\n" + "="*50)
print(f"📥 RAW INPUT:\n{messy_input}\n")

# Step 1: Regex
clean_text = layer_1_sanitize(messy_input)

# Step 2: Local Model (T5)
local_result = layer_2_dyslexia_fix(clean_text)
print(f"🧹 DYSLEXAI (Local) Output:\n{local_result}\n")

# Step 3: Groq API (Context)
final_result = get_groq_correction(local_result)

print(f"✨ FINAL RESULT:\n{final_result}")
print("="*50)


📥 RAW INPUT:
if the threatened . counter . revolution " not # to bring the President . backs . thes . 13 Stakes . of the Commonwealth was an occasion worthy . of his . presence . after . all it . was Mr. Nklymuh

🧹 DYSLEXAI (Local) Output:
If the threatened counter revolution " not to bring the President backs the 13 Stakes of the Commonwealth was an occasion worthy of his presence, after all it was Mr. Nklymuh.

📡 Contacting Groq (Llama 3.3) Layer...
✨ FINAL RESULT:
If the threatened counter-revolution "not to bring the President back to the 13 States of the Commonwealth" was an occasion worthy of his presence, after all it was Mr. Nkrumah.
